In [32]:
import pandas as pd

In [33]:
telemetry_r1_path = "barber-motorsports-park/barber/R1_barber_telemetry_data.CSV"
telemetry_r2_path = "barber-motorsports-park/barber/R2_barber_telemetry_data.CSV"

telemetry_r1 = pd.read_csv(telemetry_r1_path)
# telemetry_r2 = pd.read_csv(telemetry_r2_path)

In [35]:
telemetry_r1['vehicle_number'].unique()


array([  0,  78,   7,  16,  80,  31,  55,  13,  47,  72,  18,  46,  98,
        93,   3,  21,  88,   2, 113,   5], dtype=int64)

In [1]:
# Convert into parquet for memory optimization

import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.dataset as pads
import pyarrow.compute as pc
from pathlib import Path

telemetry_r1_parquet_path = "barber-motorsports-park/barber/R1_barber_telemetry_data.parquet"
telemetry_r2_parquet_path = "barber-motorsports-park/barber/R2_barber_telemetry_data.parquet"


#telemetry_r1.to_parquet(telemetry_r1_parquet_path, engine="pyarrow", index=False, compression="zstd", compression_level=12)
#telemetry_r2.to_parquet(telemetry_r2_parquet_path, engine="pyarrow", index=False, compression="zstd", compression_level=12)


In [5]:

telemetry_r1 = pd.read_parquet(telemetry_r1_parquet_path)

In [16]:
telemetry_r1['vehicle_number'].unique()

array([  0,  78,   7,  16,  80,  31,  55,  13,  47,  72,  18,  46,  98,
        93,   3,  21,  88,   2, 113,   5], dtype=int64)

In [29]:
telemetry_r1[(telemetry_r1['vehicle_number']==88)]

,expire_at,lap,meta_event,meta_session,meta_source,meta_time,original_vehicle_id,outing,telemetry_name,telemetry_value,timestamp,vehicle_id,vehicle_number
9420487,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:50.071Z,GR86-049-88,0,accx_can,0.218000,2025-09-04T23:48:52.688Z,GR86-049-88,88
9420488,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:50.071Z,GR86-049-88,0,accy_can,0.102000,2025-09-04T23:48:52.688Z,GR86-049-88,88
9420489,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:50.071Z,GR86-049-88,0,aps,100.000000,2025-09-04T23:48:52.688Z,GR86-049-88,88
9420490,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:50.071Z,GR86-049-88,0,pbrake_r,0.000000,2025-09-04T23:48:52.688Z,GR86-049-88,88
9420491,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:50.071Z,GR86-049-88,0,pbrake_f,0.000000,2025-09-04T23:48:52.688Z,GR86-049-88,88
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9791789,NaN,14,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T19:01:49.682Z,GR86-049-88,0,pbrake_f,0.000000,2025-09-05T00:09:58.148Z,GR86-049-88,88
9791790,NaN,14,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T19:01:49.682Z,GR86-049-88,0,gear,4.000000,2025-09-05T00:09:58.148Z,GR86-049-88,88
9791791,NaN,14,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T19:01:49.682Z,GR86-049-88,0,VBOX_Long_Minutes,-86.619560,2025-09-05T00:09:58.148Z,GR86-049-88,88
9791792,NaN,14,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T19:01:49.682Z,GR86-049-88,0,VBOX_Lat_Min,33.532669,2025-09-05T00:09:58.148Z,GR86-049-88,88


array([  0,  78,   7,  16,  80,  31,  55,  13,  47,  72,  18,  46,  98,
        93,   3,  21,  88,   2, 113,   5], dtype=int64)

In [ ]:
telemetry_r1[(telemetry_r1['vehicle_number']==13) & (telemetry_r1['lap']==14)]

NameError: name 'telemetry_r1' is not defined

In [92]:
telemetry_r1[(telemetry_r1['vehicle_number']==13) & (telemetry_r1['lap']==14)]['telemetry_name'].value_counts()

telemetry_r1[(telemetry_r1['vehicle_number']==13) & (telemetry_r1['lap']==14)]['meta_time'].min()

'2025-09-06T18:59:44.147Z'

### Create bounds for Race 1

In [ ]:
barber_start_R1_path = "barber-motorsports-park/barber/R1_barber_lap_start.CSV"
barber_end_R1_path = "barber-motorsports-park/barber/R1_barber_lap_end.CSV"
barber_lap_time_R1_path = "barber-motorsports-park/barber/R1_barber_lap_time.CSV"

# logic -> actual start we use is: later of (earliest start tick, previous lap’s end).
# if earliest start is earlier than prev lap end, we discard it as it can create negative values

def build_lap_bounds(start=None, end=None, lap=None, *, min_sec=20, max_sec=300):
    """
    Build per-lap start/end/time with vehicle_number preserved.
    Any subset of (start, end, lap) may be provided.
    """
    base_keys = ["meta_event","meta_session","vehicle_id","lap"]
    keys_with_num = base_keys + ["vehicle_number"]

    def _read(x):
        if x is None: return None
        df = x if isinstance(x, pd.DataFrame) else pd.read_csv(x)
        # coerce types
        if "lap" in df: df["lap"] = pd.to_numeric(df["lap"], errors="coerce").astype("Int64")
        if "vehicle_number" in df: df["vehicle_number"] = pd.to_numeric(df["vehicle_number"], errors="coerce").astype("Int64")
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
        # keep only expected columns if present
        keep = [c for c in keys_with_num + ["timestamp"] if c in df.columns]
        return df[keep].copy()

    st, ed, lp = _read(start), _read(end), _read(lap)

    # Normalize column names for start/end
    if st is not None: st = st.rename(columns={"timestamp":"lap_start_ts"})
    if ed is not None: ed = ed.rename(columns={"timestamp":"lap_end_ts"})
    if lp is not None: lp = lp.rename(columns={"timestamp":"lap_end_ts"})  # lap file = end crossing

    # Merge available pieces (prefer start+end if present)
    if (st is not None) or (ed is not None):
        # Build a skeleton index of laps
        idx = None
        for part in (st, ed):
            if part is not None:
                cols = [c for c in keys_with_num if c in part.columns]
                idx = part[cols].drop_duplicates() if idx is None else pd.concat([idx, part[cols]]).drop_duplicates()
        out = idx
        if st is not None: out = out.merge(st, on=[c for c in out.columns if c in st.columns and c != "lap_start_ts"], how="left")
        if ed is not None: out = out.merge(ed, on=[c for c in out.columns if c in ed.columns and c != "lap_end_ts"], how="left")
    elif lp is not None:
        out = lp.copy()
    else:
        raise ValueError("Provide at least one of: start, end, or lap")

    # Ensure mandatory keys exist
    for k in base_keys:
        if k not in out.columns: out[k] = pd.NA
    if "vehicle_number" not in out.columns: out["vehicle_number"] = pd.NA

    # Sort canonically
    sort_cols = [c for c in ["meta_event","meta_session","vehicle_id","vehicle_number","lap","lap_start_ts","lap_end_ts"] if c in out.columns]
    out = out.sort_values(sort_cols).reset_index(drop=True)

    # Group by car/session (without vehicle_number to allow backfill)
    gkeys = ["meta_event","meta_session","vehicle_id"]

    # If we have a lap file, derive missing starts from previous lap end
    if ("lap_start_ts" not in out.columns) or out["lap_start_ts"].isna().any():
        if lp is not None:
            lp2 = lp.sort_values(gkeys+["lap"])
            lp2["lap_start_ts_fromlap"] = lp2.groupby(gkeys)["lap_end_ts"].shift(1)
            out = out.merge(lp2[gkeys+["lap","lap_start_ts_fromlap"]], on=gkeys+["lap"], how="left")
            out["lap_start_ts"] = out.get("lap_start_ts", pd.NaT)
            out["lap_start_ts"] = out["lap_start_ts"].fillna(out["lap_start_ts_fromlap"])
            out = out.drop(columns=["lap_start_ts_fromlap"], errors="ignore")
        # still missing? use previous row's end within group
        out["lap_start_ts"] = out["lap_start_ts"].fillna(
            out.sort_values(gkeys+["lap"]).groupby(gkeys)["lap_end_ts"].shift(1)
        )

    # If end missing, use next start within group
    if ("lap_end_ts" not in out.columns) or out["lap_end_ts"].isna().any():
        out["lap_end_ts"] = out["lap_end_ts"].fillna(
            out.sort_values(gkeys+["lap"]).groupby(gkeys)["lap_start_ts"].shift(-1)
        )

    # Compute lap time
    out["lap_time_s"] = (out["lap_end_ts"] - out["lap_start_ts"]).dt.total_seconds()

    # Keep sane intervals
    out = out[(out["lap_time_s"].notna()) & (out["lap_time_s"] > min_sec) & (out["lap_time_s"] < max_sec)].copy()

    # Final tidy types/order
    out["lap"] = out["lap"].astype("Int64")
    out["vehicle_number"] = out["vehicle_number"].astype("Int64")
    cols_order = ["meta_event","meta_session","vehicle_id","vehicle_number","lap","lap_start_ts","lap_end_ts","lap_time_s"]
    cols_order = [c for c in cols_order if c in out.columns]
    return out[cols_order].reset_index(drop=True)



# EXAMPLES
bounds = build_lap_bounds(start = barber_start_R1_path, end = barber_end_R1_path, lap = barber_lap_time_R1_path)



In [62]:
# import pandas as pd

def dedupe_bounds(bounds, min_sec=20, max_sec=300):
    # 1) collapse duplicates to one row per lap
    agg = (bounds
           .groupby(["meta_event","meta_session","vehicle_id","vehicle_number","lap"], dropna=False)
           .agg(lap_start_ts=("lap_start_ts","min"),
                lap_end_ts  =("lap_end_ts","max"))
           .reset_index())

    # 2) ensure monotonicity (optional: clip any start earlier than previous end)
    agg = agg.sort_values(["meta_event","meta_session","vehicle_id","lap"])
    agg["lap_start_ts"] = (agg
        .groupby(["meta_event","meta_session","vehicle_id"])["lap_start_ts"]
        .cummax()
    )

    # 3) recompute lap time & keep sane intervals
    agg["lap_time_s"] = (agg["lap_end_ts"] - agg["lap_start_ts"]).dt.total_seconds()
    clean = agg[(agg["lap_time_s"].notna()) &
                (agg["lap_time_s"] > min_sec) &
                (agg["lap_time_s"] < max_sec)].copy()

    # 4) final tidy
    clean["lap"] = clean["lap"].astype("Int64")
    clean["vehicle_number"] = clean["vehicle_number"].astype("Int64")
    return clean.reset_index(drop=True)



bounds = dedupe_bounds(bounds)

In [ ]:
bounds[bounds['vehicle_number']==13]



# from start
# 2025-09-06T19:01:20.903Z
# 2025-09-06T18:59:42.903Z	

# from end
# 2025-09-06T18:59:42.902Z	
# 2025-09-06T18:59:42.902Z


	

,meta_event,meta_session,vehicle_id,vehicle_number,lap,lap_start_ts,lap_end_ts,lap_time_s
180,I_R06_2025-09-07,R1,GR86-022-13,13,3,2025-09-06 18:40:41.269000+00:00,2025-09-06 18:42:23.575000+00:00,102.306
181,I_R06_2025-09-07,R1,GR86-022-13,13,4,2025-09-06 18:42:23.576000+00:00,2025-09-06 18:44:38.610000+00:00,135.034
182,I_R06_2025-09-07,R1,GR86-022-13,13,5,2025-09-06 18:44:38.611000+00:00,2025-09-06 18:46:41.699000+00:00,123.088
183,I_R06_2025-09-07,R1,GR86-022-13,13,6,2025-09-06 18:46:41.700000+00:00,2025-09-06 18:48:19.596000+00:00,97.896
184,I_R06_2025-09-07,R1,GR86-022-13,13,7,2025-09-06 18:48:19.597000+00:00,2025-09-06 18:49:57.436000+00:00,97.839
185,I_R06_2025-09-07,R1,GR86-022-13,13,8,2025-09-06 18:49:57.437000+00:00,2025-09-06 18:51:35.376000+00:00,97.939
186,I_R06_2025-09-07,R1,GR86-022-13,13,9,2025-09-06 18:51:35.377000+00:00,2025-09-06 18:53:12.383000+00:00,97.006
187,I_R06_2025-09-07,R1,GR86-022-13,13,10,2025-09-06 18:53:12.384000+00:00,2025-09-06 18:54:49.878000+00:00,97.494
188,I_R06_2025-09-07,R1,GR86-022-13,13,11,2025-09-06 18:54:49.879000+00:00,2025-09-06 18:56:27.486000+00:00,97.607
189,I_R06_2025-09-07,R1,GR86-022-13,13,12,2025-09-06 18:56:27.487000+00:00,2025-09-06 18:58:05.180000+00:00,97.693


In [63]:
# bounds[bounds['vehicle_number']==13].to_csv('bounds-13.csv')

In [119]:
imp_cols = ['meta_session', 'vehicle_number','lap','lap_start_ts','lap_end_ts','lap_time_s']

bounds = bounds[imp_cols]

bounds.to_csv('bounds1.csv',index=False)



In [70]:
barber_start_R2_path = "barber-motorsports-park/barber/R2_barber_lap_start.CSV"
barber_end_R2_path = "barber-motorsports-park/barber/R2_barber_lap_end.CSV"
barber_lap_time_R2_path = "barber-motorsports-park/barber/R2_barber_lap_time.CSV"


start_df = pd.read_csv(barber_start_R2_path)
end_df = pd.read_csv(barber_end_R2_path)
lap_df = pd.read_csv(barber_lap_time_R2_path)

In [ ]:



bounds2 = build_lap_bounds(start = barber_start_R2_path, end = barber_end_R2_path, lap = barber_lap_time_R2_path)



In [118]:

bounds2 = bounds2[imp_cols]
bounds2.to_csv('bounds2.csv',index=False)

In [ ]:
# adhoc analysis

In [97]:
df = telemetry_r1[(telemetry_r1['vehicle_number']==13) & (telemetry_r1['lap']==14)]

In [113]:
df[df['telemetry_name']=='speed'].sort_values(by=['timestamp'], ascending= True)['telemetry_value'].value_counts()

telemetry_value
173.16    2
88.12     2
162.91    2
142.65    2
159.00    2
         ..
89.35     1
87.17     1
84.98     1
82.95     1
168.65    1
Name: count, Length: 440, dtype: int64

In [103]:
df['telemetry_name'].unique()

array(['Laptrigger_lapdist_dls', 'nmot', 'accx_can', 'accy_can', 'aps',
       'pbrake_r', 'pbrake_f', 'gear', 'VBOX_Long_Minutes',
       'VBOX_Lat_Min', 'Steering_Angle', 'speed'], dtype=object)